In [7]:
from langchain_classic.agents import create_react_agent, load_tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.agents import initialize_agent
import os
from dotenv import load_dotenv
load_dotenv()

True

#### Converstional Agent

In [16]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, api_key=os.getenv("GOOGLE_API_KEY"))
tools = load_tools(['llm-math'], llm=llm)

In [17]:
memory = ConversationBufferMemory(memory_key="chat_history")

conversational_agent = initialize_agent(
    verbose = True,
    memory=memory,
    agent="conversational-react-description",
    llm=llm,
    tools=tools,
    max_iteration= 4

)




In [18]:
output_1 = conversational_agent.run("Add 7 to the 5")
output2 = conversational_agent.run("Add 10 to the previous result")



> Entering new AgentExecutor chain...
Observation: 12
Thought: Do I need to use a tool? No
AI: The sum of 7 and 5 is 12.

> Finished chain.


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 12 + 10
Observation: Answer: 22
Thought:Do I need to use a tool? No
AI: The sum of 10 and the previous result (12) is 22.

> Finished chain.


#### ReAct Docstore

This agent utilize ReAct framework to communicate with a docstore. It requires the availabilty of a search tool and a lookup tool with identical names. The search tool is used to search for a document while the lookup tool looks up a term within the most recently found document.

docstore is a langchain storage system for documnets. use to access and interact with information

In [21]:
from langchain_classic import Wikipedia
from langchain_classic.agents.react.base import DocstoreExplorer
from langchain_classic.agents import Tool

In [24]:
docstore = DocstoreExplorer(Wikipedia())

tools = [
    Tool(name="Search", func=docstore.search, description="useful for when you need to search wikepedia"),
    Tool(name="Lookup", func=docstore.lookup, description="useful for when you need to lookup a term in wikipedia")
]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",api_key=os.getenv("GOOGLE_API_KEY"), temperature=0)
docstore_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="react-docstore",
    verbose=True,
    handle_parsing_errors = True,
    max_iteration=3
)

docstore_agent("Tell me a few key things about Mahatma Gandhi.")

C:\Users\HP\AppData\Local\Temp\ipykernel_25276\111333642.py:18: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  docstore_agent("Tell me a few key things about Mahatma Gandhi.")




> Entering new AgentExecutor chain...
Could not parse LLM Output: Mahatma Gandhi was an Indian lawyer, anti-colonial nationalist, and political ethicist. He is best known for leading India to independence from British Rule through nonviolent resistance, a philosophy he called Satyagraha. His methods and philosophy inspired civil rights and freedom movements across the world.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
Observation: Invalid or incomplete response
Thought:Could not parse LLM Output: Mahatma Gandhi was an Indian lawyer, anti-colonial nationalist, and political ethicist. He is best known for leading India to independence from British Rule through nonviolent resistance, a philosophy he called Satyagraha. His methods and philosophy inspired civil rights and freedom movements across the world.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
Observation: I

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 41.176937609s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT